In [1]:
##### This code preps the capital and labor data to be overlayed with priority maps
# converting capital and labor to priority map grid (equal 10km2) 
# because its easier to split continuous variables than the priority discrete ranks

import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

In [2]:
##### Load data

# Get the current working directory
cd = Path.cwd().parent 

# reference raster
ref_path = f"{cd}/Data/Clean/Priority_areas/biodiversity_no_PA.tif"

# capital rasters
capital = f"{cd}/Results/Raster_model/rescaled_capital_USD.tif"
capital_p10 = f"{cd}/Results/Raster_model/rescaled_capital_USD_p10.tif"
capital_p90 = f"{cd}/Results/Raster_model/rescaled_capital_USD_p90.tif"
capital_country_avg = f"{cd}/Results/Raster_model/country_avg_model/capital_USD.tif"

# labor rasters
labor = f"{cd}/Results/Raster_model/rescaled_jobs.tif"
labor_p10 = f"{cd}/Results/Raster_model/rescaled_jobs_p10.tif"
labor_p90 = f"{cd}/Results/Raster_model/rescaled_jobs_p90.tif"
labor_country_avg = f"{cd}/Results/Raster_model/country_avg_model/jobs.tif"

# production 
production = f'{cd}/Data/Clean/Production/total_production_tonnes_2020.tif'

# save paths
capital_path = f"{cd}/Results/Raster_model/reprojected/rescaled_capital_USD.tif"
capital_p10_path = f"{cd}/Results/Raster_model/reprojected/rescaled_capital_USD_p10.tif"
capital_p90_path = f"{cd}/Results/Raster_model/reprojected/rescaled_capital_USD_p90.tif"
capital_country_avg_path = f"{cd}/Results/Raster_model/reprojected/rescaled_capital_USD_country_avg.tif"

labor_path = f"{cd}/Results/Raster_model/reprojected/rescaled_jobs.tif"
labor_p10_path = f"{cd}/Results/Raster_model/reprojected/rescaled_jobs_p10.tif"
labor_p90_path = f"{cd}/Results/Raster_model/reprojected/rescaled_jobs_p90.tif"
labor_country_avg_path = f"{cd}/Results/Raster_model/reprojected/rescaled_jobs_country_avg.tif"

production_path = f"{cd}/Results/Raster_model/reprojected/total_production_tonnes_2020.tif"

In [3]:
#### Resample to match priority rasters

def resample_raster(in_path, out_path):
    with rasterio.open(ref_path) as src_a, rasterio.open(in_path) as src_in:
        dst_array = np.full((src_a.height, src_a.width), np.nan, dtype=np.float32)

        reproject(
            source=rasterio.band(src_in, 1),
            destination=dst_array,
            src_transform=src_in.transform,
            src_crs=src_in.crs,
            src_nodata=src_in.nodata,
            dst_transform=src_a.transform,
            dst_crs=src_a.crs,
            dst_nodata=np.nan,
            resampling=Resampling.sum,
        )

        out_meta = src_a.meta.copy()
        out_meta.update(dtype="float32", count=1, nodata=np.nan)

        with rasterio.open(out_path, "w", **out_meta) as dst:
            dst.write(dst_array, 1)

        total_before = np.nansum(src_in.read(1))

    total_after = np.nansum(dst_array)
    print(f"{in_path}: before={total_before:.1f}, after={total_after:.1f}, ratio={total_after/total_before:.4f}")

resample_raster(capital, capital_path)
resample_raster(capital_p10, capital_p10_path)
resample_raster(capital_p90, capital_p90_path)
resample_raster(labor, labor_path)
resample_raster(labor_p10, labor_p10_path)
resample_raster(labor_p90, labor_p90_path)
resample_raster(production, production_path) # its fine because it drops non-land vars
resample_raster(capital_country_avg, capital_country_avg_path)
resample_raster(labor_country_avg, labor_country_avg_path)

/Users/carinamanitius/Documents/GitHub/AgDownscaling/Results/Raster_model/rescaled_capital_USD.tif: before=5355437293568.0, after=5355440439296.0, ratio=1.0000
/Users/carinamanitius/Documents/GitHub/AgDownscaling/Results/Raster_model/rescaled_capital_USD_p10.tif: before=5355437817856.0, after=5355437293568.0, ratio=1.0000
/Users/carinamanitius/Documents/GitHub/AgDownscaling/Results/Raster_model/rescaled_capital_USD_p90.tif: before=5355442536448.0, after=5355433099264.0, ratio=1.0000
/Users/carinamanitius/Documents/GitHub/AgDownscaling/Results/Raster_model/rescaled_jobs.tif: before=836145216.0, after=836145024.0, ratio=1.0000
/Users/carinamanitius/Documents/GitHub/AgDownscaling/Results/Raster_model/rescaled_jobs_p10.tif: before=836144960.0, after=836145088.0, ratio=1.0000
/Users/carinamanitius/Documents/GitHub/AgDownscaling/Results/Raster_model/rescaled_jobs_p90.tif: before=836145088.0, after=836144896.0, ratio=1.0000
/Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Clean/Produ